# ✦ Celesta — Kepler Exoplanet Classification
## India High School Exoplanet Data Challenge

**Team:** Srinath V Venkateshan  
**Dataset:** NASA Exoplanet Archive — Kepler Objects of Interest (KOI) Cumulative Table  
**Task:** Classify each KOI as `CONFIRMED`, `CANDIDATE`, or `FALSE POSITIVE` using only raw transit and stellar measurements — **zero data leakage**.

---
### Table of Contents
1. Setup & Imports
2. Data Loading & Overview
3. Exploratory Data Analysis (EDA)
4. Data Cleaning & Missing-Value Strategy
5. Feature Engineering
6. Model Selection & Training
7. Evaluation (CV + Held-Out Test)
8. Explainability (SHAP)
9. Plain-Language Summary
10. Conclusion & Next Steps

---
## 1. Setup & Imports

In [ ]:
# Install required packages (run once)
# !pip install xgboost shap imbalanced-learn scikit-learn pandas numpy matplotlib seaborn

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    VotingClassifier,
    RandomForestClassifier,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

class BalancedXGBClassifier(XGBClassifier):
    """XGBoost with auto-balanced sample weights at fit time."""
    def fit(self, X, y, **kwargs):
        kwargs.setdefault("sample_weight", compute_sample_weight("balanced", y))
        return super().fit(X, y, **kwargs)


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'font.family':      'sans-serif',
})
PALETTE = {'CONFIRMED': '#4fc3f7', 'CANDIDATE': '#a5d6a7', 'FALSE POSITIVE': '#ef9a9a'}
print('✓ Setup complete')

---
## 2. Data Loading & Overview

In [ ]:
df_raw = pd.read_csv('data/koi_stripped.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
print('Target distribution:')
counts = df_raw['koi_disposition'].value_counts()
print(counts)

fig, ax = plt.subplots(figsize=(7, 4))
colors = [PALETTE[c] for c in counts.index]
bars = ax.barh(counts.index, counts.values, color=colors, edgecolor='none')
ax.bar_label(bars, padding=4, color='#e6edf3', fontsize=11)
ax.set_xlabel('Number of KOIs')
ax.set_title('Class Distribution — Kepler Objects of Interest', pad=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('docs/static/images/class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nImbalance ratio  FP : CONFIRMED : CANDIDATE = {counts["FALSE POSITIVE"]/counts["CANDIDATE"]:.1f} : {counts["CONFIRMED"]/counts["CANDIDATE"]:.1f} : 1')

---
## 3. Exploratory Data Analysis

In [ ]:
# ── Missing value heatmap ─────────────────────────────────────────────────────
miss = df_raw.isnull().mean().sort_values(ascending=False)
miss_nonzero = miss[miss > 0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(miss_nonzero.index, miss_nonzero.values * 100,
        color=['#ef9a9a' if v > 0.5 else '#ffcc80' if v > 0.1 else '#a5d6a7'
               for v in miss_nonzero.values])
ax.axvline(100, color='#ef9a9a', linewidth=1, linestyle='--', label='100% missing → dropped')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Feature', pad=12)
ax.legend()
plt.tight_layout()
plt.show()
print('Features with 100% missing (DROPPED):', miss_nonzero[miss_nonzero == 1.0].index.tolist())

In [ ]:
# ── Key feature distributions by class ───────────────────────────────────────
key_features = ['koi_model_snr', 'koi_prad', 'koi_period', 'koi_impact',
                'koi_depth', 'koi_ror', 'koi_max_sngle_ev', 'koi_max_mult_ev']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    ax = axes[i]
    for cls, color in PALETTE.items():
        subset = df_raw[df_raw['koi_disposition'] == cls][feat].dropna()
        subset_log = np.log1p(subset.clip(lower=0))
        ax.hist(subset_log, bins=40, alpha=0.55, color=color, label=cls, density=True)
    ax.set_title(f'log(1+{feat})', fontsize=9)
    ax.set_xlabel('')
    if i == 0:
        ax.legend(fontsize=7)

fig.suptitle('Feature Distributions by Class (log-scale)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation matrix (top features) ────────────────────────────────────────
top_feats = ['koi_model_snr', 'koi_max_mult_ev', 'koi_ror', 'koi_prad',
             'koi_dor', 'koi_impact', 'koi_depth', 'koi_period',
             'koi_teq', 'koi_max_sngle_ev']
corr = df_raw[top_feats].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 8}, linewidths=0.3)
ax.set_title('Feature Correlation Matrix', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: SNR vs planet radius, coloured by class ─────────────────────────
sample = df_raw.dropna(subset=['koi_model_snr', 'koi_prad', 'koi_disposition'])

fig, ax = plt.subplots(figsize=(9, 5))
for cls, color in PALETTE.items():
    sub = sample[sample['koi_disposition'] == cls]
    ax.scatter(np.log1p(sub['koi_prad']), np.log1p(sub['koi_model_snr']),
               alpha=0.25, s=10, color=color, label=cls)
ax.set_xlabel('log(1 + Planet Radius [R⊕])')
ax.set_ylabel('log(1 + Transit SNR)')
ax.set_title('Planet Radius vs Transit SNR — coloured by Disposition')
ax.legend(markerscale=2)
plt.tight_layout()
plt.show()

**Key EDA findings:**
- **FALSE POSITIVEs** tend to have higher single-event statistics relative to multi-event stats — suggesting one-off signals, not repeating transits.
- **CONFIRMED** planets cluster at lower planet radii and higher transit SNR.
- **CANDIDATE** class overlaps with both — it is genuinely ambiguous, making it the hardest to classify.
- Three features (`koi_sage`, `koi_model_dof`, `koi_model_chisq`) are **100% missing** → dropped entirely.
- `koi_bin_oedp_sig`, `koi_max_sngle_ev`, `koi_max_mult_ev` have ~12–16% missing → median imputed.

---
## 4. Data Cleaning & Missing-Value Strategy

In [ ]:
RAW_FEATURES = [
    # Transit shape
    'koi_period', 'koi_time0bk', 'koi_impact', 'koi_duration',
    'koi_depth', 'koi_ror', 'koi_srho', 'koi_prad', 'koi_sma',
    'koi_incl', 'koi_teq', 'koi_insol', 'koi_dor',
    # Signal statistics
    'koi_max_sngle_ev', 'koi_max_mult_ev', 'koi_model_snr',
    'koi_count', 'koi_num_transits', 'koi_bin_oedp_sig',
    # Stellar properties
    'koi_steff', 'koi_slogg', 'koi_smet', 'koi_srad', 'koi_smass',
    # Sky position & brightness
    'ra', 'dec', 'koi_kepmag',
    # Dropped: koi_sage (100% NaN), koi_model_dof (100% NaN), koi_model_chisq (100% NaN)
]

TARGET = 'koi_disposition'

available = [c for c in RAW_FEATURES if c in df_raw.columns]
df = df_raw[available + [TARGET]].copy()
df = df[df[TARGET].notna()].copy()

print(f'Dropped 100%-missing: koi_sage, koi_model_dof, koi_model_chisq')
print(f'Working dataset: {df.shape}')
print(f'Remaining missing (handled via median imputation in pipeline):')
m = df[available].isnull().mean()
print(m[m > 0].sort_values(ascending=False).to_string())

**Imputation strategy:**  
- Features with < 20% missing → **median imputation** (robust to outliers).  
- XGBoost and HistGradientBoosting handle `NaN` natively during training — no imputation needed for those estimators.  
- The Random Forest sub-estimator wraps a median imputer in its pipeline.  
- We deliberately avoid mean imputation because several transit features (`koi_depth`, `koi_prad`, `koi_model_snr`) are strongly right-skewed.

---
## 5. Feature Engineering

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add 8 physics-motivated derived features.

    Rationale
    ---------
    single_multi_ratio     : single-event / multi-event statistic.
                             FPs often have a high single-event outlier that
                             inflates this ratio.
    duration_period_ratio  : transit duration / orbital period (hours).
                             Encodes transit chord length and constrains
                             stellar density via Kepler's 3rd law.
    log_period/depth/snr   : log-normalise right-skewed distributions,
                             linearising decision boundaries.
    stellar_density_proxy  : M_star / R_star³ — high density (M-dwarf)
                             makes large-radius false positives less plausible.
    impact_ror_ratio       : impact parameter / radius ratio — values near 1
                             indicate grazing transits (geometry-based FP flag).
    expected_duration_ratio: observed / Kepler-geometry predicted duration.
                             Deviations flag eccentricity or blended binaries.
    """
    d = df.copy()
    eps = 1e-9

    d['single_multi_ratio']      = d['koi_max_sngle_ev'] / (d['koi_max_mult_ev'] + eps)
    d['duration_period_ratio']   = d['koi_duration'] / (d['koi_period'] * 24.0 + eps)
    d['log_period']              = np.log1p(d['koi_period'].clip(lower=0))
    d['log_depth']               = np.log1p(d['koi_depth'].clip(lower=0))
    d['log_snr']                 = np.log1p(d['koi_model_snr'].clip(lower=0))
    d['stellar_density_proxy']   = d['koi_smass'] / (d['koi_srad'] ** 3 + eps)
    d['impact_ror_ratio']        = d['koi_impact'] / (d['koi_ror'] + eps)
    expected = (
        d['koi_period'] * 24.0 / np.pi
        * d['koi_ror'] / (d['koi_dor'] + eps)
        * np.sqrt((1 - d['koi_impact'] ** 2).clip(lower=0))
    )
    d['expected_duration_ratio'] = d['koi_duration'] / (expected + eps)

    return d


ENGINEERED = [
    'single_multi_ratio', 'duration_period_ratio',
    'log_period', 'log_depth', 'log_snr',
    'stellar_density_proxy', 'impact_ror_ratio', 'expected_duration_ratio',
]

df_eng = engineer_features(df)
ALL_FEATURES = available + ENGINEERED

print(f'Raw features:        {len(available)}')
print(f'Engineered features: {len(ENGINEERED)}')
print(f'Total features:      {len(ALL_FEATURES)}')

In [ ]:
# Show that single_multi_ratio separates classes well
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (feat, title) in zip(axes, [
    ('single_multi_ratio',    'Single/Multi Event Ratio (new)'),
    ('expected_duration_ratio', 'Expected Duration Ratio (new)'),
]):
    for cls, color in PALETTE.items():
        sub = df_eng[df_eng[TARGET] == cls][feat].dropna()
        sub_clip = np.log1p(sub.clip(lower=0, upper=sub.quantile(0.99)))
        ax.hist(sub_clip, bins=50, alpha=0.55, color=color, label=cls, density=True)
    ax.set_title(title)
    ax.set_xlabel('log(1 + value)')
    ax.legend(fontsize=8)

fig.suptitle('Engineered Features — Class Separation', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. Model Selection & Training

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df_eng[TARGET])
X  = df_eng[ALL_FEATURES].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

sw_train = compute_sample_weight('balanced', y_train)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print('Classes:', list(le.classes_))

In [ ]:
# ── Ensemble: XGBoost + HistGradientBoosting + RandomForest ──────────────────
#
# Why an ensemble?
#   • XGBoost  — best on structured/tabular data, handles NaN natively
#   • HistGradBoost — scikit-learn's gradient boosting, handles NaN + class_weight
#   • RandomForest  — high variance, low correlation with boosters; diversifies
# Soft voting averages predicted probabilities — more accurate than hard voting.
# Class imbalance handled by balanced sample weights (XGBoost) and
# class_weight='balanced' (HGB, RF).

n_classes = len(le.classes_)

xgb = BalancedXGBClassifier(
    n_estimators=500, learning_rate=0.04, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    objective='multi:softprob', num_class=n_classes,
    eval_metric='mlogloss', tree_method='hist',
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
)

hgb = HistGradientBoostingClassifier(
    max_iter=500, learning_rate=0.04, max_depth=8,
    min_samples_leaf=20, l2_regularization=0.1,
    class_weight='balanced', random_state=RANDOM_STATE,
)

rf_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('clf', RandomForestClassifier(
        n_estimators=400, max_depth=16, min_samples_leaf=3,
        max_features='sqrt', class_weight='balanced_subsample',
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
])

ensemble = VotingClassifier(
    estimators=[('xgb', xgb), ('hgb', hgb), ('rf', rf_pipe)],
    voting='soft', n_jobs=1,
)

print('Training ensemble (XGBoost + HistGradBoost + RandomForest)…')
ensemble.fit(X_train, y_train)  # balanced weights applied inside BalancedXGBClassifier
print('✓ Training complete')

---
## 7. Evaluation

In [ ]:
y_pred  = ensemble.predict(X_test)
y_proba = ensemble.predict_proba(X_test)

acc         = accuracy_score(y_test, y_pred)
macro_f1    = f1_score(y_test, y_pred, average='macro')
weighted_f1 = f1_score(y_test, y_pred, average='weighted')
macro_rec   = recall_score(y_test, y_pred, average='macro')
macro_prec  = precision_score(y_test, y_pred, average='macro')

print('─' * 50)
print(f'  Accuracy      : {acc:.4f}')
print(f'  Macro F1      : {macro_f1:.4f}')
print(f'  Weighted F1   : {weighted_f1:.4f}')
print(f'  Macro Recall  : {macro_rec:.4f}')
print(f'  Macro Precision: {macro_prec:.4f}')
print('─' * 50)
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Held-Out Test Set', pad=12)
plt.tight_layout()
plt.savefig('docs/static/images/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── One-vs-Rest ROC curves ────────────────────────────────────────────────────
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

fig, ax = plt.subplots(figsize=(7, 5))
for i, (cls, color) in enumerate(PALETTE.items()):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('One-vs-Rest ROC Curves')
ax.legend()
plt.tight_layout()
plt.savefig('docs/static/images/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5-Fold Stratified Cross-Validation ───────────────────────────────────────
print('Running 5-fold CV… (this takes a few minutes)')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    ensemble, X, y,
    cv=cv, scoring='f1_macro', n_jobs=1
)
print(f'CV Macro F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('Per-fold:   ', [f'{s:.4f}' for s in cv_scores])

---
## 8. Explainability — SHAP Values

In [ ]:
# Use XGBoost sub-estimator for SHAP (TreeExplainer)
print('Computing SHAP values…')
background  = X_train.sample(min(500, len(X_train)), random_state=RANDOM_STATE)
explainer   = shap.TreeExplainer(ensemble.estimators_[0])   # XGBoost
shap_values = explainer.shap_values(background)

# For multiclass, shap_values is a list of [n_samples, n_features] per class
if isinstance(shap_values, list):
    mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    # XGBoost ≥ 2 returns 3D array [n_samples, n_features, n_classes]
    mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2)) if shap_values.ndim == 3 else np.abs(shap_values).mean(axis=0)

top_k    = 15
top_idx  = np.argsort(mean_abs_shap)[-top_k:][::-1]
top_feats_shap = [ALL_FEATURES[i] for i in top_idx]
top_vals_shap  = mean_abs_shap[top_idx]

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#4fc3f7' if f in ENGINEERED else '#a5d6a7' for f in top_feats_shap]
bars = ax.barh(top_feats_shap[::-1], top_vals_shap[::-1], color=colors[::-1])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title(f'Top {top_k} Features by SHAP Importance', pad=12)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4fc3f7', label='Engineered feature'),
    Patch(facecolor='#a5d6a7', label='Raw Kepler feature'),
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig('docs/static/images/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP summary plot per class
class_names = list(le.classes_)
if isinstance(shap_values, list) and len(shap_values) == len(class_names):
    shap.summary_plot(
        shap_values[0],     # CONFIRMED class
        background,
        feature_names=ALL_FEATURES,
        max_display=12,
        plot_type='bar',
        show=True,
        plot_size=(10, 6),
    )
else:
    print('SHAP summary plot: use shap.summary_plot(shap_values, background, ...) with your version')

---
## 9. Plain-Language Summary

### What did the model learn?

The most important signal is **transit Signal-to-Noise Ratio (`koi_model_snr`)** — how clearly the star's brightness dips when the planet passes in front. A strong, consistent dip points to a real planet. A faint or noisy dip is more likely a FALSE POSITIVE.

The second-most important feature is the **Multiple Event Statistic (`koi_max_mult_ev`)** — this measures how confidently the transit repeats across multiple orbits. Real planets repeat like clockwork; instrumental noise and background binary stars usually do not.

Our **engineered feature** `single_multi_ratio` (single-event / multi-event statistic) proved to be one of the top discriminators: when a signal spikes on just one orbit instead of repeating consistently, it's very likely a false positive — perhaps a cosmic ray or a background eclipsing binary.

The `expected_duration_ratio` (how long the transit lasts vs how long Kepler's geometry predicts it *should* last) flags unusual cases: transits that are too short or too long for the measured orbital parameters hint at eccentric orbits or blended binary stars.

### Why is CANDIDATE the hardest class?

The CANDIDATE label literally means "we don't know yet" — these are Kepler objects that passed initial screening but haven't been confirmed or ruled out by follow-up observations. The model correctly captures this ambiguity: CANDIDATE predictions have the lowest confidence scores and the most overlap with both confirmed planets and false positives.

### Leakage-free guarantee

We deliberately excluded NASA vetting outputs (`koi_pdisposition`, `koi_score`, all `koi_vet_*` columns, and text comments). Using these would be like giving the model the answer sheet — it would score 99%+ but tell us nothing about the underlying astronomy. Our 35 features are exclusively raw transit photometry and stellar measurements — the same data a real astrophysicist would use before any human review.

---
## 10. Conclusion & Next Steps

### Summary Table

| Model | Macro F1 | Accuracy | Notes |
|---|---|---|---|
| Baseline Random Forest (v1) | 0.768 | 0.792 | 25 trees, no feature engineering |
| **Celesta Ensemble (v2)** | **see above** | **see above** | XGBoost + HGB + RF, 8 engineered features |

### What we improved
- **Dropped** three 100%-missing features that added noise to the imputer.
- **Added 8 physics-motivated features**: transit duration ratio, single/multi event ratio, log-transforms, stellar density proxy, grazing transit indicator, and expected-duration deviation.
- **Upgraded the model** from a single 25-tree Random Forest to a soft-voting ensemble of XGBoost (500 trees), HistGradientBoosting (500 iter), and Random Forest (400 trees).
- **Fixed class imbalance** using balanced sample weights, improving CANDIDATE class F1.
- **Validated** with 5-fold stratified CV instead of a single split.

### Further improvements (future work)
- Bayesian hyperparameter optimisation (Optuna) for each ensemble member.
- ADASYN oversampling specifically for the CANDIDATE class.
- Neural network branch (TabNet or a shallow MLP) as a fourth ensemble member.
- Calibrate class probabilities with Platt scaling for better confidence estimates.
- Use the full Kepler DR25 TCE dataset (includes pipeline-level features) for richer training data.